In [1]:
import xarray as xr

In [2]:
ds = xr.open_zarr(
    'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3',
    chunks=None,
    storage_options={'token': 'anon'}
)

ds

<xarray.Dataset> Size: 4PB
Dimensions:                                                          (
                                                                      time: 1323648,
                                                                      latitude: 721,
                                                                      longitude: 1440,
                                                                      level: 37)
Coordinates:
  * time                                                             (time) datetime64[ns] 11MB ...
  * latitude                                                         (latitude) float32 3kB ...
  * longitude                                                        (longitude) float32 6kB ...
  * level                                                            (level) int64 296B ...
Data variables: (12/273)
    100m_u_component_of_wind                                         (time, latitude, longitude) float32 5TB ...
    100m_v_component_of_wind                                         (time, latitude, longitude) float32 5TB ...
    10m_u_component_of_neutral_wind                                  (time, latitude, longitude) float32 5TB ...
    10m_u_component_of_wind                                          (time, latitude, longitude) float32 5TB ...
    10m_v_component_of_neutral_wind                                  (time, latitude, longitude) float32 5TB ...
    10m_v_component_of_wind                                          (time, latitude, longitude) float32 5TB ...
    ...                                                               ...
    wave_spectral_directional_width_for_swell                        (time, latitude, longitude) float32 5TB ...
    wave_spectral_directional_width_for_wind_waves                   (time, latitude, longitude) float32 5TB ...
    wave_spectral_kurtosis                                           (time, latitude, longitude) float32 5TB ...
    wave_spectral_peakedness                                         (time, latitude, longitude) float32 5TB ...
    wave_spectral_skewness                                           (time, latitude, longitude) float32 5TB ...
    zero_degree_level                                                (time, latitude, longitude) float32 5TB ...
Attributes:
    last_updated:           2026-09-18 04:40:05.198736+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2026-06-30
    valid_time_stop_era5t:  2026-09-12

In [3]:
ds.nbytes / (1024 ** 4)

3344.695211229595

In [4]:
import hvplot.xarray
ds['2m_temperature'].sel(time='2020-01-01T12').hvplot()

:Image   [longitude,latitude]   (2 metre temperature)

In [5]:
(
    ds['2m_temperature']
    .sel(time=slice('2020-01-01T00', '2020-01-03T00'))
    .hvplot(groupby='time', x='longitude', y='latitude', dynamic=False,
            widget_type="scrubber", widget_location="bottom")
)

Column
    [0] HoloViews(HoloMap, height=300, sizing_mode='fixed', widget_location='bottom', widget_type='scrubber', width=700)
    [1] WidgetBox(align=('center', 'end'))
        [0] Player(end=48, width=550)

#### Xarray-SQL

In [6]:
import xarray_sql as xql

ctx = xql.XarrayContext()

# Make sure to pass chunks
ctx.from_dataset('era5', ds, chunks=dict(time=6), table_names={
    ('time', 'latitude', 'longitude'): 'surface',
    ('time', 'level', 'latitude', 'longitude'): 'atmosphere',
})

SessionContext: id=71275d83-cb95-47df-9cd5-6cf7691ecf0f; configs=[
	datafusion.catalog.create_default_catalog_and_schema = true
	datafusion.catalog.default_catalog = datafusion
	datafusion.catalog.default_schema = public
	datafusion.catalog.has_header = true
	datafusion.catalog.information_schema = true
	datafusion.catalog.newlines_in_values = false
	datafusion.execution.batch_size = 8192
	datafusion.execution.coalesce_batches = true
	datafusion.execution.collect_statistics = true
	datafusion.execution.enable_ansi_mode = false
	datafusion.execution.enable_recursive_ctes = true
	datafusion.execution.enforce_batch_size_in_joins = false
	datafusion.execution.hash_join_buffering_capacity = 0
	datafusion.execution.keep_partition_by_columns = false
	datafusion.execution.listing_table_factory_infer_partitions = true
	datafusion.execution.listing_table_ignore_subdirectory = true
	datafusion.execution.max_buffered_batches_per_output_file = 2
	datafusion.execution.max_spill_file_size_bytes = 134

In [7]:
ctx.sql("show tables")

table_catalog,table_schema,table_name,table_type
datafusion,era5,atmosphere,BASE TABLE
datafusion,era5,surface,BASE TABLE
datafusion,information_schema,tables,VIEW
datafusion,information_schema,views,VIEW
datafusion,information_schema,columns,VIEW
datafusion,information_schema,df_settings,VIEW
datafusion,information_schema,schemata,VIEW
datafusion,information_schema,routines,VIEW
datafusion,information_schema,parameters,VIEW


In [8]:
ctx.sql("show columns from era5.surface")

table_catalog,table_schema,table_name,column_name,data_type,is_nullable
datafusion,era5,surface,latitude,Float32,YES
datafusion,era5,surface,longitude,Float32,YES
datafusion,era5,surface,time,Timestamp(ns),YES
datafusion,era5,surface,100m_u_component_of_wind,Float32,YES
datafusion,era5,surface,100m_v_component_of_wind,Float32,YES
datafusion,era5,surface,10m_u_component_of_neutra10m_u_component_of_neutral_wind...,Float32,YES
datafusion,era5,surface,10m_u_component_of_wind,Float32,YES
datafusion,era5,surface,10m_v_component_of_neutra10m_v_component_of_neutral_wind...,Float32,YES
datafusion,era5,surface,10m_v_component_of_wind,Float32,YES
datafusion,era5,surface,10m_wind_gust_since_previ10m_wind_gust_since_previous_post_processing...,Float32,YES


In [9]:
ctx.sql("show columns from era5.atmosphere")

table_catalog,table_schema,table_name,column_name,data_type,is_nullable
datafusion,era5,atmosphere,latitude,Float32,YES
datafusion,era5,atmosphere,level,Int64,YES
datafusion,era5,atmosphere,longitude,Float32,YES
datafusion,era5,atmosphere,time,Timestamp(ns),YES
datafusion,era5,atmosphere,fraction_of_cloud_cover,Float32,YES
datafusion,era5,atmosphere,geopotential,Float32,YES
datafusion,era5,atmosphere,ozone_mass_mixing_ratio,Float32,YES
datafusion,era5,atmosphere,potential_vorticity,Float32,YES
datafusion,era5,atmosphere,specific_cloud_ice_water_specific_cloud_ice_water_content...,Float32,YES
datafusion,era5,atmosphere,specific_cloud_liquid_watspecific_cloud_liquid_water_content...,Float32,YES


In [10]:
ctx.sql('''
    SELECT level, AVG(temperature) - 273.15 AS avg_c
    FROM era5.atmosphere
    WHERE time BETWEEN TIMESTAMP '2020-01-01'
        AND TIMESTAMP '2020-01-01 05:00:00'
    GROUP BY level
    ORDER BY level DESC
''')

level,avg_c
1000,6.6210120796502565
975,5.185637919348153
950,4.028428657263021
925,3.0828117974912743
900,2.2109172992531967
875,1.395017610194202
850,0.6342670572626616
825,-0.21037158786759846
800,-1.1810754318269687
775,-2.3064649711534457


In [11]:
ctx.sql('''
    SELECT latitude, longitude, AVG("2m_temperature") - 273.15 AS avg_c
    FROM era5.surface
    WHERE time BETWEEN TIMESTAMP '2020-01-01'
        AND TIMESTAMP '2020-01-01 05:00:00'
    GROUP BY latitude, longitude
    ORDER BY latitude DESC, longitude
''').to_dataset(dims=['latitude', "longitude"], template=ds)

<xarray.Dataset> Size: 8MB
Dimensions:    (latitude: 721, longitude: 1440)
Coordinates:
  * latitude   (latitude) float32 3kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float32 6kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
Data variables:
    avg_c      (latitude, longitude) float64 8MB -26.84 -26.84 ... -27.38 -27.38
Attributes:
    last_updated:           2026-09-18 04:40:05.198736+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2026-06-30
    valid_time_stop_era5t:  2026-09-12

#### DuckDB + XQL

DuckDB-Zarr

In [12]:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False


In [13]:
import duckdb
import pandas as pd

%load_ext sql

conn = duckdb.connect()

%sql conn --alias duckdb

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

In [14]:
%%sql

INSTALL zarr FROM community;
LOAD zarr;

select name, dims, shape
from read_zarr_metadata('gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3')

,name,dims,shape
0,100m_u_component_of_wind,"[""time"",""latitude"",""longitude""]","[1323648,721,1440]"
1,100m_v_component_of_wind,"[""time"",""latitude"",""longitude""]","[1323648,721,1440]"
2,10m_u_component_of_neutral_wind,"[""time"",""latitude"",""longitude""]","[1323648,721,1440]"
3,10m_u_component_of_wind,"[""time"",""latitude"",""longitude""]","[1323648,721,1440]"
4,10m_v_component_of_neutral_wind,"[""time"",""latitude"",""longitude""]","[1323648,721,1440]"
...,...,...,...
272,wave_spectral_directional_width_for_wind_waves,"[""time"",""latitude"",""longitude""]","[1323648,721,1440]"
273,wave_spectral_kurtosis,"[""time"",""latitude"",""longitude""]","[1323648,721,1440]"
274,wave_spectral_peakedness,"[""time"",""latitude"",""longitude""]","[1323648,721,1440]"
275,wave_spectral_skewness,"[""time"",""latitude"",""longitude""]","[1323648,721,1440]"


In [15]:
# The community `zarr` extension can inspect store metadata, but it currently
# cannot decode the data chunks of this store, so value queries would return
# fill values. To query real values with DuckDB, materialize a small slice
# with xarray and hand it to DuckDB through the Arrow columnar interface.
subset = (
    ds["2m_temperature"]
    .sel(time="2020-02-02T00")
    .sel(latitude=slice(42.0, 32.5), longitude=slice(235.5, 245.9))
    .load()
)
subset_df = subset.to_dataframe().reset_index()

In [16]:
%%sql
select time, latitude, longitude, "2m_temperature" - 273.15 AS t2_c
from subset_df
order by latitude desc, longitude
limit 10

,time,latitude,longitude,t2_c
0,2020-02-02,42.0,235.50,10.419556
1,2020-02-02,42.0,235.75,10.109863
2,2020-02-02,42.0,236.00,8.683777
3,2020-02-02,42.0,236.25,7.695709
4,2020-02-02,42.0,236.50,8.026031
5,2020-02-02,42.0,236.75,8.539246
6,2020-02-02,42.0,237.00,9.034760
7,2020-02-02,42.0,237.25,9.621704
8,2020-02-02,42.0,237.50,11.221802
9,2020-02-02,42.0,237.75,11.926727


#### Round-tripping Query Results to Xarray

The DuckDB backend of xarray-sql was removed in version 0.3 — the package now
ships a DataFusion backend, so the old `xarray_sql.register()` /
`xarray_sql.to_dataset()` helpers no longer exist.

Instead we reuse the `XarrayContext` (`ctx`) registered in the first section.
The query below pulls average `2m_temperature` over California for a 6-hour
window. The original query used a `ST_Within` polygon, but that polygon was an
axis-aligned rectangle, so a bounding box is exactly equivalent. Note that ERA5
longitude runs from 0 to 360°, so California is around 235–246°E. Finally the
result is round-tripped back to an `xr.Dataset` with
`XarrayDataFrame.to_dataset`, using the original dataset as the metadata
template.

In [17]:
rel = ctx.sql("""
    SELECT time, latitude, longitude,
           AVG("2m_temperature") - 273.15 AS avg_c
    FROM era5.surface
    WHERE longitude BETWEEN 235.5 AND 245.9
      AND latitude BETWEEN 32.5 AND 42.0
      AND time BETWEEN TIMESTAMP '2020-02-02' AND TIMESTAMP '2020-02-02 05:00:00'
    GROUP BY time, latitude, longitude
    ORDER BY time, latitude DESC, longitude
""")
rel

time,latitude,longitude,avg_c
2020-02-02 00:00:00,42.0,235.5,10.419549560546898
2020-02-02 00:00:00,42.0,235.75,10.109857177734398
2020-02-02 00:00:00,42.0,236.0,8.683770751953148
2020-02-02 00:00:00,42.0,236.25,7.695703125000023
2020-02-02 00:00:00,42.0,236.5,8.026025390625023
2020-02-02 00:00:00,42.0,236.75,8.539239501953148
2020-02-02 00:00:00,42.0,237.0,9.034753417968773
2020-02-02 00:00:00,42.0,237.25,9.621697998046898
2020-02-02 00:00:00,42.0,237.5,11.221795654296898
2020-02-02 00:00:00,42.0,237.75,11.926721191406273


In [18]:
ca_ds = rel.to_dataset(
    dims=['time', 'latitude', 'longitude'], template=ds, chunks=None
)
ca_ds

<xarray.Dataset> Size: 79kB
Dimensions:    (time: 6, latitude: 39, longitude: 42)
Coordinates:
  * time       (time) datetime64[ns] 48B 2020-02-02 ... 2020-02-02T05:00:00
  * latitude   (latitude) float32 156B 42.0 41.75 41.5 41.25 ... 33.0 32.75 32.5
  * longitude  (longitude) float32 168B 235.5 235.8 236.0 ... 245.2 245.5 245.8
Data variables:
    avg_c      (time, latitude, longitude) float64 79kB 10.42 10.11 ... 18.77
Attributes:
    last_updated:           2026-09-18 04:40:05.198736+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2026-06-30
    valid_time_stop_era5t:  2026-09-12